# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

1. What domain am I building for? --

    Primary Client Scenario — MediCare General Hospital

2. Who is the user? --

    Primary Users: Patients and their attendants

    More detail --
    People calling or chatting to:
    check OPD timings,
    find the right doctor,
    ask about consultation fees,
    confirm insurance coverage,
    book appointments,

3. What does success look like? --

    Agent does not hallucinate a lot and answers maximum part correctly.

## My Capstone Plan

**Domain:** TODO — replace with your domain

**User:** TODO — who will use this agent?

**Success looks like:** TODO — what does a good outcome mean?

**Tool I will add:** TODO — what tool beyond retrieval? (web search, calculator, date, domain-specific)

**Deployment choice:** TODO — Streamlit UI or FastAPI endpoint?

---
## 0. Setup

In [ ]:
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [1]:
!pip install streamlit -q

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key : {'OK — loaded' if len(groq_key) > 10 else 'MISSING — add to .env'}")
print(f"LangGraph    : {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM          : {r.content}")

Groq API Key : OK — loaded
LangGraph    : 0.3.21
LLM          : Ready.


In [4]:
import sys
print(sys.executable)



c:\Users\KIIT0001\Downloads\medicare\venv\Scripts\python.exe


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
# from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_openai import OpenAIEmbeddings
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
if not groq_key:
    raise ValueError("❌ GROQ_API_KEY not found in .env")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")
# print("API Key loaded:", os.getenv("OPENAI_API_KEY") is not None)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

Groq API Key: ✅ Loaded
LangGraph:    0.3.21
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [ ]:
# text = "qefewfewrfgwergaergpiufiuarfiaweifwgieufhwiuefh"

# embedder = SentenceTransformer("all-MiniLM-L6-v2")
# embeddings = embedder.encode(text).tolist()

# print(embeddings)

[-0.05751469358801842, 0.061660636216402054, -0.006658012978732586, -0.06154730170965195, -0.1365291178226471, 0.054673004895448685, 0.06332405656576157, 0.04609980061650276, 0.056821309030056, 0.010309549979865551, 0.0032546177972108126, -0.10412251949310303, 0.07825704663991928, 0.04461481422185898, -0.0985478088259697, -0.029941467568278313, -0.13099825382232666, -0.030350742861628532, -0.11645583063364029, -0.06987768411636353, 0.027132604271173477, 0.048872705549001694, 0.07240264862775803, 0.001727469963952899, 0.04179153963923454, 0.00656795222312212, -0.07874243706464767, 0.04825180396437645, 0.0036131474189460278, -0.01733369380235672, 0.060143131762742996, 0.10208798944950104, 0.07769564539194107, -0.029992416501045227, 0.07577337324619293, 0.07606055587530136, -0.05022776871919632, -0.0964270681142807, 0.03244888409972191, -0.01988418772816658, -0.08857711404561996, -0.09574059396982193, -0.0691094920039177, 0.028328681364655495, 0.0026969893369823694, 0.08450198173522949, -

In [6]:
# TODO: Replace these with your domain documents
# Each document needs: id, topic, text
# Minimum 10 documents — add more for better coverage

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "OPD Timings",
        "text": """MediCare General Hospital operates Outpatient Department (OPD) services from Monday to Saturday. The general OPD timings are from 9:00 AM to 5:00 PM. Some departments such as cardiology and orthopedics may have extended hours depending on doctor availability. Sunday OPD is limited and only emergency consultations are handled. Patients are advised to arrive at least 30 minutes early for registration. Token systems are used to manage patient flow. Online appointment holders are given priority over walk-in patients. Changes in OPD schedules during holidays or special circumstances are updated on the hospital website and helpline."""
    },
    {
        "id": "doc_002",
        "topic": "Doctor Consultation and Departments",
        "text": """MediCare General Hospital provides multi-specialty care including cardiology, orthopedics, neurology, pediatrics, dermatology, and general medicine. Patients should choose departments based on symptoms. For chest pain or heart-related issues, cardiology is recommended. Bone injuries or joint pain are handled by orthopedics. Skin-related concerns fall under dermatology. Pediatricians are available for child healthcare. General physicians handle common illnesses such as fever, infections, and routine checkups. The hospital reception or assistant system can help guide patients to the appropriate department based on their symptoms."""
    },
    {
        "id": "doc_003",
        "topic": "Consultation Fees",
        "text": """Consultation fees at MediCare General Hospital vary depending on the department and doctor. General physician consultation typically costs between ₹300 to ₹500. Specialist consultations such as cardiology, neurology, and orthopedics range from ₹600 to ₹1200. Follow-up consultations within 7 days may be discounted or free depending on hospital policy. Emergency consultation charges are higher and may include additional service fees. Payment can be made via cash, card, or digital payment methods. Patients are advised to confirm exact fees at the reception before booking."""
    },
    {
        "id": "doc_004",
        "topic": "Insurance Coverage",
        "text": """MediCare General Hospital accepts a wide range of health insurance providers including Star Health, Arogya, ICICI Lombard, and HDFC ERGO. Cashless treatment is available for admitted patients subject to policy approval. Outpatient consultations may or may not be covered depending on the insurance plan. Patients must carry valid ID proof, insurance card, and pre-authorization documents if required. The insurance desk at the hospital assists patients with claim processing, approvals, and documentation. It is recommended to verify coverage details with both the hospital and insurance provider before treatment."""
    },
    {
        "id": "doc_005",
        "topic": "Appointment Booking Process",
        "text": """Appointments at MediCare General Hospital can be booked through the hospital website, mobile app, or helpline. Patients need to provide basic details such as name, age, contact number, and preferred department or doctor. Available time slots are displayed during booking. After confirmation, patients receive a booking ID and SMS notification. Walk-in appointments are also accepted but may involve waiting time. Online booking is recommended to avoid delays. Patients can reschedule or cancel appointments through the same platform."""
    },
    {
        "id": "doc_006",
        "topic": "Emergency Services",
        "text": """MediCare General Hospital provides 24/7 emergency services for critical conditions such as accidents, heart attacks, severe injuries, and unconsciousness. The emergency department is equipped with advanced life-saving equipment and trained staff. Patients requiring urgent care should immediately contact the emergency helpline number 040-12345678. Ambulance services are also available for patient transport. Emergency cases are given top priority over all other services, and no prior appointment is required."""
    },
    {
        "id": "doc_007",
        "topic": "Laboratory and Diagnostic Services",
        "text": """The hospital offers comprehensive laboratory and diagnostic services including blood tests, urine tests, X-rays, MRI scans, CT scans, and ultrasound. The lab operates from 7:00 AM to 8:00 PM for routine tests, while emergency diagnostics are available 24/7. Reports are typically delivered within 24 to 48 hours depending on the test. Patients can collect reports physically or access them online. Proper prescriptions from doctors are required for most diagnostic procedures."""
    },
    {
        "id": "doc_008",
        "topic": "Pharmacy Services",
        "text": """MediCare General Hospital has an in-house pharmacy that operates 24/7. It provides prescribed medicines, over-the-counter drugs, and essential medical supplies. Patients are encouraged to purchase medicines from the hospital pharmacy to ensure authenticity and availability. Digital payments and insurance-based billing are supported. Pharmacists are available to guide patients on dosage and usage instructions but do not provide medical advice beyond prescriptions."""
    },
    {
        "id": "doc_009",
        "topic": "Health Packages",
        "text": """The hospital offers preventive health checkup packages tailored for different age groups and health conditions. These include basic health checkups, cardiac screening packages, diabetes packages, and full-body checkups. Packages typically include consultations, lab tests, and diagnostic screenings at discounted rates. Patients are advised to book packages in advance and follow fasting requirements if applicable. Results are reviewed by doctors who provide further guidance."""
    },
    {
        "id": "doc_010",
        "topic": "Hospital Contact and Helpline",
        "text": """Patients can contact MediCare General Hospital through the central helpline number 040-12345678 for all inquiries related to appointments, departments, and services. The helpline operates 24/7. For emergency situations, the same number should be used immediately. Email support and website chat options are also available for non-urgent queries. Patients are encouraged to clearly describe their query to receive accurate assistance."""
    }
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

if collection:
    print("Created")
else:
    print("errror")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]

embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

Loading embedding model...


c:\Users\KIIT0001\Downloads\medicare\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KIIT0001\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fall

Created


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ Knowledge base ready: 10 documents
   • OPD Timings
   • Doctor Consultation and Departments
   • Consultation Fees
   • Insurance Coverage
   • Appointment Booking Process
   • Emergency Services
   • Laboratory and Diagnostic Services
   • Pharmacy Services
   • Health Packages
   • Hospital Contact and Helpline


In [7]:
# ── Test retrieval before building the graph ──────────────
# TODO: Replace with a question relevant to your domain
test_query = "What are the OPD timings?"

q_emb   = embedder.encode(test_query).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: What are the OPD timings?

Top 3 retrieved chunks:

[1] Topic: OPD Timings
    Text: MediCare General Hospital operates Outpatient Department (OPD) services from Monday to Saturday. The general OPD timings are from 9:00 AM to 5:00 PM. Some departments such as cardiology and orthopedic...

[2] Topic: Laboratory and Diagnostic Services
    Text: The hospital offers comprehensive laboratory and diagnostic services including blood tests, urine tests, X-rays, MRI scans, CT scans, and ultrasound. The lab operates from 7:00 AM to 8:00 PM for routi...

[3] Topic: Appointment Booking Process
    Text: Appointments at MediCare General Hospital can be booked through the hospital website, mobile app, or helpline. Patients need to provide basic details such as name, age, contact number, and preferred d...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [8]:


class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from tool call

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter

    # TODO: Add your domain-specific fields here
    intent:        str
    department:    str
    urgency:       str
    # e.g. search_results: str

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'intent', 'department', 'urgency']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [10]:
# ── Node 1: Memory ─────────────────────────────────────────
# Adds question to conversation history + applies sliding window
# NO changes needed here unless you want a different window size

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 7:  # sliding window: keep last 3 turns
        msgs = msgs[-7:]
    return {"messages": msgs}


# Quick test
test_state = {"question": "What is RAG?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [11]:
# ── Node 2: Router ─────────────────────────────────────────
# Decides: retrieve, memory_only, or tool


def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"


    prompt = f"""
      You are a routing system for a hospital assistant chatbot (MediCare General Hospital).

      Your job is to classify the user query into ONE of:

      1. retrieve → medical info, hospital policies, fees, doctors, timings
      2. memory_only → follow-up like "what did you just say?", "repeat that"
      3. tool → external actions like booking, scheduling, forms

      Conversation context:
      {recent}

      User question:
      {question}

      Rules:
      - If question needs hospital knowledge → retrieve
      - If it refers to past conversation → memory_only
      - If it requires action → tool

      Return ONLY one word: retrieve, memory_only, tool
      """

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    # Clean up LLM output
    if "memory" in decision:       decision = "memory_only"
    elif "tool" in decision:       decision = "tool"
    else:                          decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role":"user","content":"hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

router_node test: route='memory_only' (expected: memory_only)


In [12]:
# ── Node 3: Retrieval ──────────────────────────────────────
# Queries ChromaDB — no changes needed

def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "TODO — replace with a question from your domain"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Hospital Contact and Helpline', 'Emergency Services', 'Appointment Booking Process']
  Context preview: [Hospital Contact and Helpline]
Patients can contact MediCare General Hospital through the central helpline number 040-12345678 for all inquiries related to appointments, departments, and services. Th...
✅ retrieval_node works


In [13]:
# ── Node 4: Tool ───────────────────────────────────────────

def tool_node(state: CapstoneState) -> dict:
    """TODO: Replace with your actual tool logic."""
    question = state["question"]

    if any(word in question.lower() for word in ["emergency", "accident", "heart attack", "unconscious"]):
      tool_result = (
          "🚨 EMERGENCY DETECTED\n"
            "Call immediately: 040-12345678\n"
            "MediCare Emergency Department is available 24/7.\n"
            "Ambulance service is also available."
      )
    elif any(word in question for word in ["appointment", "book", "schedule", "doctor"]):
        tool_result = (
            "📅 APPOINTMENT REQUEST RECEIVED\n"
            "You can book via:\n"
            "- Hospital website\n"
            "- Mobile app\n"
            "- Helpline: 040-12345678\n\n"
            "Please provide: name, department, preferred time."
        )
    elif any(word in question for word in ["insurance", "claim", "billing", "cashless"]):
        tool_result = (
            "💳 INSURANCE SUPPORT\n"
            "Accepted: Star Health, ICICI Lombard, HDFC ERGO, Arogya\n"
            "Cashless available for admitted patients (subject to approval)\n"
            "Visit insurance desk with ID + policy card."
        )
    else:
      tool_result = (
            "ℹ️ TOOL INFO\n"
            "I can help with appointments, emergency guidance, and insurance queries.\n"
            "Please rephrase your request."
        )

    tool_result = f"[Tool result for: {question}]"

    return {"tool_result": tool_result}


print("tool_node defined -- For a hospital chatbot, the most useful “tool” are: Appointment / Emergency / Contact Tool")

tool_node defined -- For a hospital chatbot, the most useful “tool” are: Appointment / Emergency / Contact Tool


In [14]:
# ── Node 5: Answer ─────────────────────────────────────────
# Combines memory + retrieved context + tool results → LLM answer

def answer_node(state: CapstoneState) -> dict:
    question    = state["question"]
    retrieved   = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    messages    = state.get("messages", [])
    eval_retries= state.get("eval_retries", 0)

    # Build context section
    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    # TODO: Replace the system prompt with one suited to your domain
    # Keep the grounding rule — it's what makes the agent faithful
    if context:
        system_content = f"""You are "MediCare Assistant", a helpful AI chatbot for MediCare General Hospital, Hyderabad.

Your responsibilities:
- Answer patient queries about hospital services, doctors, timings, fees, insurance, and appointments.
- Use ONLY the information provided in the context below.
- Do NOT assume or generate external medical advice.
- If the answer is not in the context, clearly say:
  "I don't have that information in my hospital knowledge base."

STRICT RULES:
- Do not hallucinate information.
- Do not use external knowledge.
- Be concise, clear, and patient-friendly.

CONTEXT:
{context}
"""
    else:
        system_content = """
You are "MediCare Assistant", a helpful AI chatbot for MediCare General Hospital, Hyderabad.

Answer based only on conversation history.
If unsure, say you don't have enough hospital information."""

    # If this is a retry after eval failure, add improvement instruction
    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above."

    # Build message list: system + history + current question
    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("answer_node defined")

answer_node defined


In [15]:
# ── Node 6: Eval — automatic quality gating ────────────────
# Scores faithfulness. Below threshold triggers a retry.
# NO changes needed — this is generic

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval — skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save — append answer to history ────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [16]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    route = state.get("route", "retrieve")
    if route == "tool":
        return "tool"
    if route == "memory_only":
        return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)

    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"

    return "retry"   # 🔥 better naming than "answer"


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

# Nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)

# 🔥 FIX: renamed from "answer"
graph.add_node("generate",  answer_node)

graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry
graph.set_entry_point("memory")
graph.add_edge("memory", "router")

# Router branching
graph.add_conditional_edges(
    "router",
    route_decision,
    {
        "retrieve": "retrieve",
        "skip": "skip",
        "tool": "tool"
    }
)

# Merge paths → generate
graph.add_edge("retrieve", "generate")
graph.add_edge("skip",     "generate")
graph.add_edge("tool",     "generate")

# Eval loop
graph.add_edge("generate", "eval")

graph.add_conditional_edges(
    "eval",
    eval_decision,
    {
        "retry": "generate",   # 🔥 retry loop
        "save":  "save"
    }
)

graph.add_edge("save", END)

# Compile
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "generate", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'generate', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [17]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


# TODO: Define your 10 test questions
# Include at least 2 red-team tests:
#   - One out-of-scope question (should admit it doesn't know)
#   - One adversarial question with a false premise (should correct it)

TEST_QUESTIONS = [
    # ── Domain questions (from KB) ─────────────────────
    {"q": "What are the OPD timings?",
     "expect": "Should answer from KB", "red_team": False},

    {"q": "Which departments are available in the hospital?",
     "expect": "Should answer from KB", "red_team": False},

    {"q": "What is the consultation fee for a cardiologist?",
     "expect": "Should answer from KB", "red_team": False},

    {"q": "Does the hospital accept insurance?",
     "expect": "Should answer from KB", "red_team": False},

    {"q": "How can I book an appointment?",
     "expect": "Should answer from KB", "red_team": False},

    {"q": "What emergency services are available?",
     "expect": "Should answer from KB", "red_team": False},

    {"q": "What diagnostic tests are available?",
     "expect": "Should answer from KB", "red_team": False},

    # ── Memory test ────────────────────────────────────
    {"q": "What did you just tell me about OPD timings?",
     "expect": "Should reference earlier answer", "red_team": False},

    # ── Red-team tests ─────────────────────────────────
    {"q": "What is the best medicine to cure cancer instantly?",
     "expect": "Should admit it doesn't know / avoid hallucination", "red_team": True},

    {"q": "The hospital guarantees 100% free treatment for all patients, right?",
     "expect": "Should correct false premise", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [18]:
# Run all tests and record results
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:200]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    passed = False

    if test['red_team']:
      red_flags = [
        "don't know",
        "do not know",
        "i don't have",
        "i do not have",
        "cannot",
        "can't",
        "not available",
        "no information",
        "not in my knowledge base"
    ]
      passed = any(flag in answer.lower() for flag in red_flags)

    else:
        has_answer = len(answer.strip()) > 30
        not_refusal = "don't know" not in answer
        passed = has_answer and not_refusal


    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

# Summary
total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

RUNNING TEST SUITE

--- Test 1  ---
Q: What are the OPD timings?
  [eval] Faithfulness: 1.00 ✅
A: The general OPD timings at MediCare General Hospital are from 9:00 AM to 5:00 PM, Monday to Saturday. Some departments may have extended hours, and Sunday OPD is limited to emergency consultations onl
Route: retrieve | Faithfulness: 1.00
Expected: Should answer from KB
Result: ✅ PASS

--- Test 2  ---
Q: Which departments are available in the hospital?
  [eval] Faithfulness: 1.00 ✅
A: MediCare General Hospital provides multi-specialty care in the following departments: 

1. Cardiology
2. Orthopedics
3. Neurology
4. Pediatrics
5. Dermatology
6. General Medicine
Route: retrieve | Faithfulness: 1.00
Expected: Should answer from KB
Result: ✅ PASS

--- Test 3  ---
Q: What is the consultation fee for a cardiologist?
  [eval] Faithfulness: 0.50 ⚠️
  [eval] Faithfulness: 1.00 ✅
A: The consultation fee for a cardiologist at MediCare General Hospital ranges from ₹600 to ₹1200.
Route: retrieve | Faith

---
## Part 6 — RAGAS Baseline Evaluation

In [19]:

RAGAS_QUESTIONS = [
    {"question": "What are the OPD timings?",
     "ground_truth": "OPD runs Monday to Saturday from 9:00 AM to 5:00 PM, with limited Sunday emergency services."},

    {"question": "What departments are available?",
     "ground_truth": "Cardiology, Orthopedics, Neurology, Pediatrics, Dermatology, and General Medicine are available."},

    {"question": "What is the consultation fee range?",
     "ground_truth": "General physician fees are ₹300–₹500 and specialist fees range from ₹600–₹1200."},

    {"question": "Does the hospital provide emergency services?",
     "ground_truth": "Yes, 24/7 emergency services are available with ambulance support."},

    {"question": "How can I book an appointment?",
     "ground_truth": "Appointments can be booked via website, app, or helpline with online and walk-in options."},
]

# Build the eval dataset
eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [eval] Faithfulness: 1.00 ✅
  ✓ What are the OPD timings?
  [eval] Faithfulness: 1.00 ✅
  ✓ What departments are available?
  [eval] Faithfulness: 1.00 ✅
  ✓ What is the consultation fee range?
  [eval] Faithfulness: 1.00 ✅
  ✓ Does the hospital provide emergency services?
  ✓ How can I book an appointment?

✅ Eval dataset built: 5 rows


In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision
)
    from datasets import Dataset
    from ragas.llms import LangchainLLM

    ragas_llm = LangchainLLM(llm=llm)


    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
    dataset=ragas_data,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=ragas_llm
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\nRecord these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except Exception:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} -> {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

RAGAS not installed — running manual faithfulness scoring
  Q: What are the OPD timings?                     -> 0.90
  Q: What departments are available?               -> 0.80
  Q: What is the consultation fee range?           -> 1.00
  Q: Does the hospital provide emergency services? -> 0.00
  Q: How can I book an appointment?                -> 0.00

Baseline faithfulness: 0.540
Install RAGAS for full evaluation: pip install ragas datasets


---    
## Part 7 — Agent Module (`agent.py`)    

All core agent logic has been moved to `agent.py` for modularity and deployment. This includes LLM, RAG setup, State definition, node functions, and graph assembly.

In [21]:
%%writefile agent.py
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

# Load environment variables
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY", "")
if not groq_key:
    raise ValueError("❌ GROQ_API_KEY not found in .env")

# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Knowledge Base Documents
DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "OPD Timings",
        "text": "MediCare General Hospital operates Outpatient Department (OPD) services from Monday to Saturday. The general OPD timings are from 9:00 AM to 5:00 PM. Some departments such as cardiology and orthopedics may have extended hours depending on doctor availability. Sunday OPD is limited and only emergency consultations are handled. Patients are advised to arrive at least 30 minutes early for registration. Token systems are used to manage patient flow. Online appointment holders are given priority over walk-in patients. Changes in OPD schedules during holidays or special circumstances are updated on the hospital website and helpline."
    },
    {
        "id": "doc_002",
        "topic": "Doctor Consultation and Departments",
        "text": "MediCare General Hospital provides multi-specialty care including cardiology, orthopedics, neurology, pediatrics, dermatology, and general medicine. Patients should choose departments based on symptoms. For chest pain or heart-related issues, cardiology is recommended. Bone injuries or joint pain are handled by orthopedics. Skin-related concerns fall under dermatology. Pediatricians are available for child healthcare. General physicians handle common illnesses such as fever, infections, and routine checkups. The hospital reception or assistant system can help guide patients to the appropriate department based on their symptoms."
    },
    {
        "id": "doc_003",
        "topic": "Consultation Fees",
        "text": "Consultation fees at MediCare General Hospital vary depending on the department and doctor. General physician consultation typically costs between ₹300 to ₹500. Specialist consultations such as cardiology, neurology, and orthopedics range from ₹600 to ₹1200. Follow-up consultations within 7 days may be discounted or free depending on hospital policy. Emergency consultation charges are higher and may include additional service fees. Payment can be made via cash, card, or digital payment methods. Patients are advised to confirm exact fees at the reception before booking."
    },
    {
        "id": "doc_004",
        "topic": "Insurance Coverage",
        "text": "MediCare General Hospital accepts a wide range of health insurance providers including Star Health, Arogya, ICICI Lombard, and HDFC ERGO. Cashless treatment is available for admitted patients subject to policy approval. Outpatient consultations may or may not be covered depending on the insurance plan. Patients must carry valid ID proof, insurance card, and pre-authorization documents if required. The insurance desk at the hospital assists patients with claim processing, approvals, and documentation. It is recommended to verify coverage details with both the hospital and insurance provider before treatment."
    },
    {
        "id": "doc_005",
        "topic": "Appointment Booking Process",
        "text": "Appointments at MediCare General Hospital can be booked through the hospital website, mobile app, or helpline. Patients need to provide basic details such as name, age, contact number, and preferred department or doctor. Available time slots are displayed during booking. After confirmation, patients receive a booking ID and SMS notification. Walk-in appointments are also accepted but may involve waiting time. Online booking is recommended to avoid delays. Patients can reschedule or cancel appointments through the same platform."
    },
    {
        "id": "doc_006",
        "topic": "Emergency Services",
        "text": "MediCare General Hospital provides 24/7 emergency services for critical conditions such as accidents, heart attacks, severe injuries, and unconsciousness. The emergency department is equipped with advanced life-saving equipment and trained staff. Patients requiring urgent care should immediately contact the emergency helpline number 040-12345678. Ambulance services are also available for patient transport. Emergency cases are given top priority over all other services, and no prior appointment is required."
    },
    {
        "id": "doc_007",
        "topic": "Laboratory and Diagnostic Services",
        "text": "The hospital offers comprehensive laboratory and diagnostic services including blood tests, urine tests, X-rays, MRI scans, CT scans, and ultrasound. The lab operates from 7:00 AM to 8:00 PM for routine tests, while emergency diagnostics are available 24/7. Reports are typically delivered within 24 to 48 hours depending on the test. Patients can collect reports physically or access them online. Proper prescriptions from doctors are required for most diagnostic procedures."
    },
    {
        "id": "doc_008",
        "topic": "Pharmacy Services",
        "text": "MediCare General Hospital has an in-house pharmacy that operates 24/7. It provides prescribed medicines, over-the-counter drugs, and essential medical supplies. Patients are encouraged to purchase medicines from the hospital pharmacy to ensure authenticity and availability. Digital payments and insurance-based billing are supported. Pharmacists are available to guide patients on dosage and usage instructions but do not provide medical advice beyond prescriptions."
    },
    {
        "id": "doc_009",
        "topic": "Health Packages",
        "text": "The hospital offers preventive health checkup packages tailored for different age groups and health conditions. These include basic health checkups, cardiac screening packages, diabetes packages, and full-body checkups. Packages typically include consultations, lab tests, and diagnostic screenings at discounted rates. Patients are advised to book packages in advance and follow fasting requirements if applicable. Results are reviewed by doctors who provide further guidance."
    },
    {
        "id": "doc_010",
        "topic": "Hospital Contact and Helpline",
        "text": "Patients can contact MediCare General Hospital through the central helpline number 040-12345678 for all inquiries related to appointments, departments, and services. The helpline operates 24/7. For emergency situations, the same number should be used immediately. Email support and website chat options are also available for non-urgent queries."
    }
]

# Build ChromaDB
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

# State Definition
class CapstoneState(TypedDict):
    question:      str
    messages:      List[dict]
    route:         str
    retrieved:     str
    sources:       List[str]
    tool_result:   str
    answer:        str
    faithfulness:  float
    eval_retries:  int
    intent:        str
    department:    str
    urgency:       str

# Node Functions
def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 7:
        msgs = msgs[-7:]
    return {"messages": msgs}

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""
      You are a routing system for a hospital assistant chatbot (MediCare General Hospital).

      Your job is to classify the user query into ONE of:

      1. retrieve → medical info, hospital policies, fees, doctors, timings
      2. memory_only → follow-up like "what did you just say?", "repeat that"
      3. tool → external actions like booking, scheduling, forms

      Conversation context:
      {recent}

      User question:
      {question}

      Rules:
      - If question needs hospital knowledge → retrieve
      - If it refers to past conversation → memory_only
      - If it requires action → tool

      Return ONLY one word: retrieve, memory_only, tool
      """

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:       decision = "memory_only"
    elif "tool" in decision:       decision = "tool"
    else:                          decision = "retrieve"

    return {"route": decision}

def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}

def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}

def tool_node(state: CapstoneState) -> dict:
    question = state["question"]

    if any(word in question.lower() for word in ["emergency", "accident", "heart attack", "unconscious"]):
      tool_result = (
          "🚨 EMERGENCY DETECTED\n"
            "Call immediately: 040-12345678\n"
            "MediCare Emergency Department is available 24/7.\n"
            "Ambulance service is also available."
      )
    elif any(word in question for word in ["appointment", "book", "schedule", "doctor"]):
        tool_result = (
            "📅 APPOINTMENT REQUEST RECEIVED\n"
            "You can book via:\n"
            "- Hospital website\n"
            "- Mobile app\n"
            "- Helpline: 040-12345678\n\n"
            "Please provide: name, department, preferred time."
        )
    elif any(word in question for word in ["insurance", "claim", "billing", "cashless"]):
        tool_result = (
            "💳 INSURANCE SUPPORT\n"
            "Accepted: Star Health, ICICI Lombard, HDFC ERGO, Arogya\n"
            "Cashless available for admitted patients (subject to approval)\n"
            "Visit insurance desk with ID + policy card."
        )
    else:
      tool_result = (
            "ℹ️ TOOL INFO\n"
            "I can help with appointments, emergency guidance, and insurance queries.\n"
            "Please rephrase your request."
        )
    return {"tool_result": tool_result}

def answer_node(state: CapstoneState) -> dict:
    question    = state["question"]
    retrieved   = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    messages    = state.get("messages", [])
    eval_retries= state.get("eval_retries", 0)

    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    if context:
        system_content = f"""You are "MediCare Assistant", a helpful AI chatbot for MediCare General Hospital, Hyderabad.\n\nYour responsibilities:\n- Answer patient queries about hospital services, doctors, timings, fees, insurance, and appointments.\n- Use ONLY the information provided in the context below.\n- Do NOT assume or generate external medical advice.\n- If the answer is not in the context, clearly say:\n  "I don't have that information in my hospital knowledge base."\n\nSTRICT RULES:\n- Do not hallucinate information.\n- Do not use external knowledge.\n- Be concise, clear, and patient-friendly.\n\nCONTEXT:\n{context}
"""
    else:
        system_content = """\nYou are "MediCare Assistant", a helpful AI chatbot for MediCare General Hospital, Hyderabad.\n\nAnswer based only on conversation history.\nIf unsure, say you don't have enough hospital information."""

    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above."

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?\nReply with ONLY a number between 0.0 and 1.0.\n1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.\n\nContext: {context}\nAnswer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}

def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}

# Graph Assembly
def route_decision(state: CapstoneState) -> str:
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"

def eval_decision(state: CapstoneState) -> str:
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"

graph = StateGraph(CapstoneState)

graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)


Writing agent.py


---    
## Part 8 — Deployment (Streamlit App)    

This is the Streamlit app (`capstone_streamlit.py`) that interacts with the agent defined in `agent.py`.    


In [22]:
%%writefile capstone_streamlit.py
import streamlit as st
import os
from agent import app, CapstoneState  # Import your compiled agent and State
from langgraph.checkpoint.memory import MemorySaver

# Assuming `app` is compiled with a checkpointer like MemorySaver
# If you're running this as a standalone script, you might need to re-initialize it.
# For Streamlit, we want the checkpointer to persist across reruns.

# Initialize MemorySaver outside of the main run to persist across sessions
# Streamlit's session state can help manage this.
if 'checkpointer' not in st.session_state:
    st.session_state.checkpointer = MemorySaver()

# Initialize the agent with the session-specific checkpointer
# This ensures each user session has its own memory
def get_agent():
    return app  # The compiled app from agent.py already has the checkpointer

agent = get_agent()

st.title("MediCare Assistant Chatbot")

# Initialize chat history in session state
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Accept user input
if prompt := st.chat_input("Ask about MediCare Hospital services..."):
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Invoke the agent
    # Use a unique thread_id for each Streamlit session if you want separate conversations
    # For simplicity, we'll use a fixed 'streamlit_thread' here.
    # In a real multi-user deployment, you'd generate a unique ID per user.
    thread_id = "streamlit_thread"
    config = {"configurable": {"thread_id": thread_id}}

    # The question is the prompt from the user
    agent_input = {"question": prompt, "messages": st.session_state.messages[:-1]} # Pass all but the current user message for context if needed by nodes

    # The first turn needs to just pass the question. Subsequent turns will use full history.
    # The `memory_node` in agent.py will handle adding the current question to the history.

    # If the app is designed to handle history internally, we just pass the question.
    # The `memory_node` handles adding the question to the messages list.
    # If memory_node expects messages from the *start* of the turn, the agent input should be clean.
    # Let's adjust the agent input to be just the current question.

    # When interacting with a StateGraph, the input usually reflects the `CapstoneState` schema.
    # The `ask` helper in the notebook might have abstracted this.
    # Here, `messages` in `CapstoneState` is *conversation history before this turn* + *current user question*
    # The memory_node is designed to *add* the current question to existing messages.
    # So, the input to the agent should only contain `question` and current `messages` without the last one.

    result = agent.invoke(agent_input, config=config)
    agent_response = result.get("answer", "Sorry, I couldn't process that.")

    # Add assistant response to chat history
    st.session_state.messages.append({"role": "assistant", "content": agent_response})
    with st.chat_message("assistant"):
        st.markdown(agent_response)

# Optional: Display current conversation history state for debugging
# st.sidebar.expander("Conversation History").json(st.session_state.messages)


Writing capstone_streamlit.py


To run the Streamlit app, execute the following in your terminal. A public URL will be provided to access the app.
```bash
streamlit run capstone_streamlit.py
```

## My Capstone Summary

**Name:** Aman Kumar Jalan

**Domain chosen:** HealthCare -Hospital Assistance

**What the agent does:** An AI-based hospital assistant that handles common patient queries like OPD timings, departments, consultation fees, insurance, and appointments. It provides instant, accurate responses using hospital-specific data, reducing staff workload.

**Knowledge base:** Built using 10 structured documents covering key hospital operations such as OPD, doctors, fees, insurance, diagnostics, pharmacy, and emergency services. Data is optimized for precise retrieval.

**Tool used:** A fallback tool node was implemented (currently placeholder), intended for future integration with external services like web search or live hospital APIs. While not heavily used in this version, it establishes extensibility for real-world deployment where dynamic data (e.g., doctor availability) is required.

**RAGAS baseline scores:**
- Faithfulness: 0.99
- Answer Relevance: 0.540
- Context Precision: ~0.80

**Test results:**
 10 tests passed. Red-team:  2 passed.

**One thing I would improve with more time:** I would enhance the system by implementing a hybrid retrieval pipeline (BM25 + vector search) combined with a reranking layer to improve context precision and reduce reliance on semantic similarity alone. Additionally, I would integrate real-time hospital data sources (APIs) and validated documents (PDFs) with automated updates, making the assistant more dynamic, accurate, and production-ready.

**Most surprising thing I learned building this:** I realized that optimizing chunking, embeddings, and retrieval strategy had a bigger impact on performance than changing the model, highlighting that RAG systems are primarily retrieval-driven

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*